In [558]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [559]:
df = pd.read_csv('rec_merged_apartments.csv')
df.head()

,apartment_name,zone,bhk_type,construction_status,carpet_area,bulit_area,super_bulit_area,price_value,nearbylocation,facility,luxury_facility_scores,luxury_category,Latitude,Longitude,apartment_loc,price_per_sqft,price_range,area_range
0,21st Castle Green Boulevard,east,3,Under Construction,NaN,1019.0,1134.00,0.67,"[('indus international school', '230 m'), ('sa...","['Squash Court', 'Yoga/Meditation Area', 'Bask...",51.0,medium,12.8925,77.7795,Sarjapur Road,5908.289242,50L-1Cr,1000-1500 sqft
1,2Gtula,east,3,Under Construction,NaN,1666.0,1854.00,1.52,"[('valistus international school', '5.9km'), (...","['Lounge', 'Creche/Day Care', ""Children's Play...",104.0,medium,12.9875,77.7085,Hoodi,8198.489752,1.5Cr-2Cr,1500-2000 sqft
2,A Grade,north,3,Under Construction,NaN,2157.0,2400.00,3.50,"['Mosque', 'Corporation bank ATM', 'Canara ban...","['Centrally Air Conditioned', 'Water purifier'...",84.0,medium,13.0365,77.5925,Hebbal,14583.333330,> 2Cr,2000-2500 sqft
3,ACS Meghana and Shali Apartments,south,2,Old,1372.0,1630.0,1837.01,1.50,"['Karnataka bank ATM', 'Syndicate bank ATM', '...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",136.0,medium,12.9412,77.5577,Banashankari Stage 2,8165.442758,1Cr-1.5Cr,1500-2000 sqft
4,AECS Layout RWA,north,4,Moderatly Old,2200.0,2614.0,2945.98,2.00,"['Sri Ramanjaneya Swamy Temple', 'Hdfc bank AT...","['Security / Fire Alarm', 'Lift(s)', 'Water St...",99.0,medium,13.0179,77.5766,Sanjayanagar,6788.912348,1.5Cr-2Cr,> 2500 sqft


In [560]:
df.columns

Index(['apartment_name', 'zone', 'bhk_type', 'construction_status',
       'carpet_area', 'bulit_area', 'super_bulit_area', 'price_value',
       'nearbylocation', 'facility', 'luxury_facility_scores',
       'luxury_category', 'Latitude', 'Longitude', 'apartment_loc',
       'price_per_sqft', 'price_range', 'area_range'],
      dtype='object')

In [561]:
df_sub = merged_df [['apartment_name','apartment_loc', 'facility']]

In [562]:
df_sub = df_sub.drop_duplicates(subset=['apartment_name','apartment_loc', 'facility'])

In [563]:
df_sub.loc[0][1]

C:\Users\amrpu\AppData\Local\Temp\ipykernel_21360\2782449505.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_sub.loc[0][1]


'Sarjapur Road'

In [564]:
df_sub

,apartment_name,apartment_loc,facility
0,21st Castle Green Boulevard,Sarjapur Road,"['Squash Court', 'Yoga/Meditation Area', 'Bask..."
1,2Gtula,Hoodi,"['Lounge', 'Creche/Day Care', ""Children's Play..."
2,A Grade,Hebbal,"['Centrally Air Conditioned', 'Water purifier'..."
3,ACS Meghana and Shali Apartments,Banashankari Stage 2,"['Security / Fire Alarm', 'Feng Shui / Vaastu ..."
4,AECS Layout RWA,Sanjayanagar,"['Security / Fire Alarm', 'Lift(s)', 'Water St..."
...,...,...,...
1470,YMR Lichen,Kothanur,"['Feng Shui / Vaastu Compliant', 'Security / F..."
1471,Yafa Zaifar,HBR Layout,"['Feng Shui / Vaastu Compliant', 'Security / F..."
1472,Yuva Sunrise,Attibele,"['Security / Fire Alarm', 'Power Back-up', 'Fe..."
1473,Yuva Utsav,Electronics City Phase 2,"['Feng Shui / Vaastu Compliant', 'Security / F..."


In [565]:
df_sub.drop_duplicates(subset=['apartment_name','apartment_loc'])

,apartment_name,apartment_loc,facility
0,21st Castle Green Boulevard,Sarjapur Road,"['Squash Court', 'Yoga/Meditation Area', 'Bask..."
1,2Gtula,Hoodi,"['Lounge', 'Creche/Day Care', ""Children's Play..."
2,A Grade,Hebbal,"['Centrally Air Conditioned', 'Water purifier'..."
3,ACS Meghana and Shali Apartments,Banashankari Stage 2,"['Security / Fire Alarm', 'Feng Shui / Vaastu ..."
4,AECS Layout RWA,Sanjayanagar,"['Security / Fire Alarm', 'Lift(s)', 'Water St..."
...,...,...,...
1470,YMR Lichen,Kothanur,"['Feng Shui / Vaastu Compliant', 'Security / F..."
1471,Yafa Zaifar,HBR Layout,"['Feng Shui / Vaastu Compliant', 'Security / F..."
1472,Yuva Sunrise,Attibele,"['Security / Fire Alarm', 'Power Back-up', 'Fe..."
1473,Yuva Utsav,Electronics City Phase 2,"['Feng Shui / Vaastu Compliant', 'Security / F..."


In [567]:
import ast
# Function to safely parse the string representation of lists
def parse_facility_list(facility_str):
    """
    Safely parse the string representation of a list of facilities
    Handles both single and double quotes and strips whitespace
    """

    # Try using ast.literal_eval for safe parsing
    parsed_list = ast.literal_eval(facility_str)
    # Strip whitespace from each item
    return [item.replace(' ', '') for item in parsed_list]



# Parse the strings into actual Python lists
facilities = df_sub['facility'].apply(parse_facility_list).tolist()

# Convert lists to strings for TF-IDF processing
# This step is necessary because TfidfVectorizer expects text documents
facility_docs = [' '.join(facility) for facility in facilities]

In [569]:
facility_docs

["SquashCourt Yoga/MeditationArea BasketballCourt ClubHouse Children'sPlayArea SwimmingPool JoggingTrack LawnTennisCourt Gymnasium",
 "Lounge Creche/DayCare Children'sPlayArea SwimmingPool Park BanquetHall JoggingTrack Theatre Gymnasium",
 'CentrallyAirConditioned Waterpurifier Security/FireAlarm FengShui/VaastuCompliant PrivateGarden/Terrace IntercomFacility Lift(s) HighCeilingHeight MaintenanceStaff FalseCeilingLighting WaterStorage Separateentryforservantroom Noopendrainagearound Piped-gas Internet/wi-ficonnectivity RecentlyRenovated VisitorParking SwimmingPool Park NaturalLight AiryRooms SpaciousInteriors WasteDisposal RainWaterHarvesting Watersofteningplant ShoppingCentre FitnessCentre/GYM',
 'Security/FireAlarm FengShui/VaastuCompliant IntercomFacility Lift(s) MaintenanceStaff WaterStorage Noopendrainagearound Piped-gas VisitorParking SwimmingPool Park SecurityPersonnel NaturalLight Internet/wi-ficonnectivity AiryRooms SpaciousInteriors FitnessCentre/GYM WasteDisposal RainWaterHa

In [570]:
vectorizer = TfidfVectorizer(lowercase=False)  # Keeping original case to preserve exact amenity names
tfidf_matrix = vectorizer.fit_transform(facility_docs)

In [571]:
# Get feature names (amenities)
feature_names = vectorizer.get_feature_names_out()

# Create a DataFrame with TF-IDF scores
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

In [572]:
df_tfidf

,24,24x7Security,7PowerBackup,7WaterSupply,AerobicsCentre,AiryRooms,Amphitheatre,BadmintonCourt,BankAttachedProperty,BanquetHall,...,WaterStorage,Waterpurifier,Watersofteningplant,Wi,Yoga,ficonnectivity,gas,sPlayArea,up,wi
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,0.320165,0.000000,0.000000,0.235110,0.000000,0.000000
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.338651,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.245647,0.000000,0.000000
2,0.0,0.0,0.0,0.0,0.0,0.160267,0.0,0.0,0.0,0.000000,...,0.107399,0.243352,0.169030,0.0,0.000000,0.172618,0.172388,0.000000,0.000000,0.172618
3,0.0,0.0,0.0,0.0,0.0,0.232615,0.0,0.0,0.0,0.000000,...,0.155881,0.000000,0.000000,0.0,0.000000,0.250541,0.250207,0.000000,0.000000,0.250541
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.458274,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.251725,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1471,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.260260,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1472,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.221535,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.346431,0.000000
1473,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.362066,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [573]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [574]:
cosine_sim1.shape

(1475, 1475)

In [575]:
def recommend_properties(property_name, cosine_sim=cosine_sim1):
    # Get the index of the property that matches the name
    idx = df_sub[df_sub['apartment_name'] == property_name].index[0]

    # Get the pairwise similarity scores with that property
    sim_scores = list(enumerate(cosine_sim1[idx]))

    # Sort the properties based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar properties
    sim_scores = sim_scores[1:6]

    # Get the property indices
    property_indices = [i[0] for i in sim_scores]
    
    recommendations_df = pd.DataFrame({
        'PropertyName': df_sub['apartment_name'].iloc[property_indices],
        'SimilarityScore': sim_scores
    })

    # Return the top 10 most similar properties
    return recommendations_df

In [576]:
recommend_properties('MJR Pearl')

,PropertyName,SimilarityScore
778,Mythri Arteor,"(778, 0.9123292648721808)"
207,Brigade Golden Triangle,"(207, 0.9010274077785041)"
966,Purva Midtown Residences,"(966, 0.8813361044776822)"
401,Fortuna Center Park,"(401, 0.8490177775591441)"
1280,Sobha Althea,"(1280, 0.8490177775591441)"


In [577]:
df_sub_2 = df[['apartment_name','apartment_loc','bhk_type','bulit_area','price_value']]

In [578]:
df_sub_2 = df_sub_2.drop_duplicates(subset=['apartment_name','apartment_loc'])


In [579]:
df_sub_2  = df_sub_2.drop(columns=['apartment_loc'])

In [580]:
df_sub_2.head()


,apartment_name,bhk_type,bulit_area,price_value
0,21st Castle Green Boulevard,3,1019.0,0.67
1,2Gtula,3,1666.0,1.52
2,A Grade,3,2157.0,3.50
3,ACS Meghana and Shali Apartments,2,1630.0,1.50
4,AECS Layout RWA,4,2614.0,2.00


In [581]:
# First, let's pivot the built_area
pivot_area = df_sub_2.pivot_table(
    index='apartment_name', 
    columns='bhk_type', 
    values='bulit_area',
    aggfunc='first'  # Use 'first' as aggregation function
).reset_index()

In [582]:
pivot_area

bhk_type,apartment_name,1,2,3,4
0,21st Castle Green Boulevard,NaN,NaN,1019.0,NaN
1,2Gtula,NaN,NaN,1666.0,NaN
2,A Grade,NaN,NaN,2157.0,NaN
3,ACS Meghana and Shali Apartments,NaN,1630.0,NaN,NaN
4,AECS Layout RWA,NaN,NaN,NaN,2614.0
...,...,...,...,...,...
1470,YMR Lichen,NaN,1044.0,NaN,NaN
1471,Yafa Zaifar,NaN,NaN,2950.0,NaN
1472,Yuva Sunrise,924.0,NaN,NaN,NaN
1473,Yuva Utsav,NaN,NaN,1302.0,NaN


In [583]:

pivot_area.columns = ['apartment_name'] + [f'area_{col}_bhk' for col in pivot_area.columns if col != 'apartment_name']

In [584]:
pivot_area

,apartment_name,area_1_bhk,area_2_bhk,area_3_bhk,area_4_bhk
0,21st Castle Green Boulevard,NaN,NaN,1019.0,NaN
1,2Gtula,NaN,NaN,1666.0,NaN
2,A Grade,NaN,NaN,2157.0,NaN
3,ACS Meghana and Shali Apartments,NaN,1630.0,NaN,NaN
4,AECS Layout RWA,NaN,NaN,NaN,2614.0
...,...,...,...,...,...
1470,YMR Lichen,NaN,1044.0,NaN,NaN
1471,Yafa Zaifar,NaN,NaN,2950.0,NaN
1472,Yuva Sunrise,924.0,NaN,NaN,NaN
1473,Yuva Utsav,NaN,NaN,1302.0,NaN


In [585]:
# Now pivot the price_value
pivot_price = df_sub_2.pivot_table(
    index='apartment_name', 
    columns='bhk_type', 
    values='price_value',
    aggfunc='first'  # Use 'first' as aggregation function
).reset_index()

# Rename the columns to include "price_" prefix for bhk_type columns
pivot_price.columns = ['apartment_name'] + [f'price_{col}_bhk' for col in pivot_price.columns if col != 'apartment_name']

In [586]:
# Merge the pivoted dataframes
result = pd.merge(pivot_area, pivot_price.drop('apartment_name', axis=1), left_index=True, right_index=True)

print("\nPivoted dataframe (wide format):")



Pivoted dataframe (wide format):


In [587]:
result


,apartment_name,area_1_bhk,area_2_bhk,area_3_bhk,area_4_bhk,price_1_bhk,price_2_bhk,price_3_bhk,price_4_bhk
0,21st Castle Green Boulevard,NaN,NaN,1019.0,NaN,NaN,NaN,0.67,NaN
1,2Gtula,NaN,NaN,1666.0,NaN,NaN,NaN,1.52,NaN
2,A Grade,NaN,NaN,2157.0,NaN,NaN,NaN,3.50,NaN
3,ACS Meghana and Shali Apartments,NaN,1630.0,NaN,NaN,NaN,1.50,NaN,NaN
4,AECS Layout RWA,NaN,NaN,NaN,2614.0,NaN,NaN,NaN,2.0
...,...,...,...,...,...,...,...,...,...
1470,YMR Lichen,NaN,1044.0,NaN,NaN,NaN,0.75,NaN,NaN
1471,Yafa Zaifar,NaN,NaN,2950.0,NaN,NaN,NaN,2.20,NaN
1472,Yuva Sunrise,924.0,NaN,NaN,NaN,0.39,NaN,NaN,NaN
1473,Yuva Utsav,NaN,NaN,1302.0,NaN,NaN,NaN,0.65,NaN


In [588]:
result=result.fillna(0)

In [589]:
result = result.set_index('apartment_name')

In [590]:
from sklearn.preprocessing import StandardScaler

# Compute the cosine similarity matrix

scaler = StandardScaler()
result_scaled = scaler.fit_transform(result)




In [591]:
# Convert the scaled array back to a Pandas DataFrame
result_scaled_df = pd.DataFrame(result_scaled, columns=result.columns, index=result.index)

# Display the DataFrame
result_scaled_df.head()

,area_1_bhk,area_2_bhk,area_3_bhk,area_4_bhk,price_1_bhk,price_2_bhk,price_3_bhk,price_4_bhk
apartment_name,,,,,,,,
21st Castle Green Boulevard,-0.279855,-0.997728,0.518237,-0.215819,-0.242603,-0.837263,0.052283,-0.198834
2Gtula,-0.279855,-0.997728,1.292956,-0.215819,-0.242603,-0.837263,0.844325,-0.198834
A Grade,-0.279855,-0.997728,1.880881,-0.215819,-0.242603,-0.837263,2.689317,-0.198834
ACS Meghana and Shali Apartments,-0.279855,1.797423,-0.701917,-0.215819,-0.242603,1.756143,-0.572033,-0.198834
AECS Layout RWA,-0.279855,-0.997728,-0.701917,3.468330,-0.242603,-0.837263,-0.572033,1.629536


In [592]:
cosine_sim2 = cosine_similarity(result_scaled)

In [593]:
cosine_sim2.shape

(1475, 1475)

In [594]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim2[result_scaled_df.index.get_loc(property_name)]))

    print(sim_scores)
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = result_scaled_df.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df

In [595]:
recommend_properties_with_scores('MJR Pearl')

[(0, np.float64(-0.046518679866498415)), (1, np.float64(-0.12778840890901041)), (2, np.float64(-0.15556279272792445)), (3, np.float64(-0.29731903049018676)), (4, np.float64(-0.013939926508218703)), (5, np.float64(-0.08015175338461893)), (6, np.float64(-0.2774556830185229)), (7, np.float64(-0.05934204103957134)), (8, np.float64(-0.29402789288236836)), (9, np.float64(-0.24065119802549406)), (10, np.float64(-0.2899669036986905)), (11, np.float64(-0.2900379433370583)), (12, np.float64(-0.2821607122138421)), (13, np.float64(-0.09393929204525117)), (14, np.float64(-0.13214992751908924)), (15, np.float64(-0.24527330346875606)), (16, np.float64(-0.2904401325970519)), (17, np.float64(-0.2775003868855756)), (18, np.float64(0.8621537299915025)), (19, np.float64(-0.07210220519309773)), (20, np.float64(-0.21663192336425108)), (21, np.float64(-0.07286147106688577)), (22, np.float64(-0.04726228682330042)), (23, np.float64(-0.19099439111356323)), (24, np.float64(-0.0612314644393318)), (25, np.float64(

,PropertyName,SimilarityScore
0,Prestige Temple Bells,0.999986
1,Godrej Nurture,0.999931
2,SNN Raj Greenbay,0.999872
3,TVS Emerald Jard,0.999845
4,Eden Park at The Prestige City,0.999647
...,...,...
242,Aarna Aplounder,-0.072102
243,Aarna Splendor,-0.072861
244,Radiant Spencer,-0.073253
245,Aratt Milano,-0.073255


## based on nearby loc

In [618]:
df_sub_3 = pd.read_csv('rec_loc_merged_apartments.csv')

In [619]:
df_sub_3.head()

,apartment_name,nearbylocation
0,21st Castle Green Boulevard,"[('Indus International School', '230 m'), ('Az..."
1,2Gtula,"[('Valistus International School', '5.9 km'), ..."
2,A Grade,"[('Baptist Hospital', '1.5 km'), ('Poornima Ho..."
3,ACS Meghana and Shali Apartments,"[('Devagiri Hospital', '1.5 km'),('BNM Institu..."
4,AECS Layout RWA,"[('RMV Hospital', '1.5 km'), ('S. J. Hospital'..."


In [620]:
df_sub_3.shape

(1475, 2)

In [621]:
df_sub_3['nearbylocation'][0]

"[('Indus International School', '230 m'), ('Azim Premji University', '1.5 km'), ('Pandana Hospital', '5.6 km'), ('Heelalige Railway Station', '16.9 km'), ('Attibele Road', '3.2 km'), ('D Mart', '5.9 km'), ('Wipro Limited', '13.2 km')]"

In [622]:
import ast

def safe_literal_eval(val):
	try:
		return ast.literal_eval(val)
	except Exception:
		return None  # or [] if you prefer an empty list

df_sub_3['nearbylocation']=df_sub_3['nearbylocation'].apply(safe_literal_eval)

In [623]:
df_sub_3=df_sub_3.explode('nearbylocation')

In [624]:
df_sub_3

,apartment_name,nearbylocation
0,21st Castle Green Boulevard,"(Indus International School, 230 m)"
0,21st Castle Green Boulevard,"(Azim Premji University, 1.5 km)"
0,21st Castle Green Boulevard,"(Pandana Hospital, 5.6 km)"
0,21st Castle Green Boulevard,"(Heelalige Railway Station, 16.9 km)"
0,21st Castle Green Boulevard,"(Attibele Road, 3.2 km)"
...,...,...
1474,Zen Indraprastha by Pratham,"(Yeshwantpur Metro Station, 1.5 km)"
1474,Zen Indraprastha by Pratham,"(Orion Mall, 2.0 km)"
1474,Zen Indraprastha by Pratham,"(National Public School, 1.8 km)"
1474,Zen Indraprastha by Pratham,"(Manipal Hospital, 2.0 km)"


In [625]:
# df_expanded = df_sub_3.explode('nearbylocation')
df_sub_3['location_name'] = df_sub_3['nearbylocation'].apply(lambda x: x[0])
df_sub_3['distance'] = df_sub_3['nearbylocation'].apply(lambda x: x[1])

In [626]:
df_sub_3

,apartment_name,nearbylocation,location_name,distance
0,21st Castle Green Boulevard,"(Indus International School, 230 m)",Indus International School,230 m
0,21st Castle Green Boulevard,"(Azim Premji University, 1.5 km)",Azim Premji University,1.5 km
0,21st Castle Green Boulevard,"(Pandana Hospital, 5.6 km)",Pandana Hospital,5.6 km
0,21st Castle Green Boulevard,"(Heelalige Railway Station, 16.9 km)",Heelalige Railway Station,16.9 km
0,21st Castle Green Boulevard,"(Attibele Road, 3.2 km)",Attibele Road,3.2 km
...,...,...,...,...
1474,Zen Indraprastha by Pratham,"(Yeshwantpur Metro Station, 1.5 km)",Yeshwantpur Metro Station,1.5 km
1474,Zen Indraprastha by Pratham,"(Orion Mall, 2.0 km)",Orion Mall,2.0 km
1474,Zen Indraprastha by Pratham,"(National Public School, 1.8 km)",National Public School,1.8 km
1474,Zen Indraprastha by Pratham,"(Manipal Hospital, 2.0 km)",Manipal Hospital,2.0 km


In [627]:
def distance_converter(x):
    if 'km' in x.lower():
        return float(x.replace('km',''))*1000
    elif 'm' in x.lower():
        return float(x.replace('m','').strip())*1.0
    else:
        return 0


In [628]:
df_sub_3['distance']=df_sub_3['distance'].apply(distance_converter)

In [629]:
df_sub_3

,apartment_name,nearbylocation,location_name,distance
0,21st Castle Green Boulevard,"(Indus International School, 230 m)",Indus International School,230.0
0,21st Castle Green Boulevard,"(Azim Premji University, 1.5 km)",Azim Premji University,1500.0
0,21st Castle Green Boulevard,"(Pandana Hospital, 5.6 km)",Pandana Hospital,5600.0
0,21st Castle Green Boulevard,"(Heelalige Railway Station, 16.9 km)",Heelalige Railway Station,16900.0
0,21st Castle Green Boulevard,"(Attibele Road, 3.2 km)",Attibele Road,3200.0
...,...,...,...,...
1474,Zen Indraprastha by Pratham,"(Yeshwantpur Metro Station, 1.5 km)",Yeshwantpur Metro Station,1500.0
1474,Zen Indraprastha by Pratham,"(Orion Mall, 2.0 km)",Orion Mall,2000.0
1474,Zen Indraprastha by Pratham,"(National Public School, 1.8 km)",National Public School,1800.0
1474,Zen Indraprastha by Pratham,"(Manipal Hospital, 2.0 km)",Manipal Hospital,2000.0


In [630]:
# Pivot the table using pivot_table to handle duplicates
df_sub_3=df_sub_3.pivot_table(
	index='apartment_name',
	columns='location_name',
	values='distance'
).reset_index()
df_sub_3

location_name,apartment_name,1 MG - Lido Mall,100 Feet Road,100 Feet Road Indiranagar,A.P.S. College of Engineering,AECS Maaruti College of Dental Sciences,AGS Layout Bus Stand,AI Vidya First Grade College,AIMS Institutes,AKASH Medical College,...,Yeshwantpur Market,Yeshwantpur Metro Station,Yesvantpur Bypass Station,Yesvantpur Junction,Yesvantpur Junction Railway Station,Yuvalok School,Zaitoon,Zeena English School,Zion Hospital And Research Centre,Zion Hospital and Research Centre
0,21st Castle Green Boulevard,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2Gtula,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A Grade,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ACS Meghana and Shali Apartments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AECS Layout RWA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,YMR Lichen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1471,Yafa Zaifar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1472,Yuva Sunrise,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1473,Yuva Utsav,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [631]:
df_sub_3.columns

Index(['apartment_name', '1 MG - Lido Mall', '100 Feet Road',
       '100 Feet Road Indiranagar', 'A.P.S. College of Engineering',
       'AECS Maaruti College of Dental Sciences', 'AGS Layout Bus Stand',
       'AI Vidya First Grade College', 'AIMS Institutes',
       'AKASH Medical College',
       ...
       'Yeshwantpur Market', 'Yeshwantpur Metro Station',
       'Yesvantpur Bypass Station', 'Yesvantpur Junction',
       'Yesvantpur Junction Railway Station', 'Yuvalok School', 'Zaitoon',
       'Zeena English School', 'Zion Hospital And Research Centre',
       'Zion Hospital and Research Centre'],
      dtype='object', name='location_name', length=2073)

In [632]:
df_sub_3=df_sub_3.set_index('apartment_name')

In [633]:
df_sub_3.head()

location_name,1 MG - Lido Mall,100 Feet Road,100 Feet Road Indiranagar,A.P.S. College of Engineering,AECS Maaruti College of Dental Sciences,AGS Layout Bus Stand,AI Vidya First Grade College,AIMS Institutes,AKASH Medical College,AMC Engineering College,...,Yeshwantpur Market,Yeshwantpur Metro Station,Yesvantpur Bypass Station,Yesvantpur Junction,Yesvantpur Junction Railway Station,Yuvalok School,Zaitoon,Zeena English School,Zion Hospital And Research Centre,Zion Hospital and Research Centre
apartment_name,,,,,,,,,,,,,,,,,,,,,
21st Castle Green Boulevard,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2Gtula,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
A Grade,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ACS Meghana and Shali Apartments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AECS Layout RWA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [634]:
df_sub_3 = df_sub_3.fillna(0)

In [635]:
from sklearn.preprocessing import StandardScaler
# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
location_df_normalized = pd.DataFrame(scaler.fit_transform(df_sub_3), columns=df_sub_3.columns, index=df_sub_3.index)

In [636]:
location_df_normalized

location_name,1 MG - Lido Mall,100 Feet Road,100 Feet Road Indiranagar,A.P.S. College of Engineering,AECS Maaruti College of Dental Sciences,AGS Layout Bus Stand,AI Vidya First Grade College,AIMS Institutes,AKASH Medical College,AMC Engineering College,...,Yeshwantpur Market,Yeshwantpur Metro Station,Yesvantpur Bypass Station,Yesvantpur Junction,Yesvantpur Junction Railway Station,Yuvalok School,Zaitoon,Zeena English School,Zion Hospital And Research Centre,Zion Hospital and Research Centre
apartment_name,,,,,,,,,,,,,,,,,,,,,
21st Castle Green Boulevard,-0.026047,-0.049266,-0.026047,-0.026047,-0.026047,-0.052146,-0.026047,-0.045087,-0.026047,-0.050994,...,-0.045145,-0.067665,-0.026047,-0.026047,-0.036848,-0.071673,-0.026047,-0.026047,-0.026047,-0.058321
2Gtula,-0.026047,-0.049266,-0.026047,-0.026047,-0.026047,-0.052146,-0.026047,-0.045087,-0.026047,-0.050994,...,-0.045145,-0.067665,-0.026047,-0.026047,-0.036848,-0.071673,-0.026047,-0.026047,-0.026047,-0.058321
A Grade,-0.026047,-0.049266,-0.026047,-0.026047,-0.026047,-0.052146,-0.026047,-0.045087,-0.026047,-0.050994,...,-0.045145,-0.067665,-0.026047,-0.026047,-0.036848,-0.071673,-0.026047,-0.026047,-0.026047,-0.058321
ACS Meghana and Shali Apartments,-0.026047,-0.049266,-0.026047,-0.026047,-0.026047,-0.052146,-0.026047,-0.045087,-0.026047,-0.050994,...,-0.045145,-0.067665,-0.026047,-0.026047,-0.036848,-0.071673,-0.026047,-0.026047,-0.026047,-0.058321
AECS Layout RWA,-0.026047,-0.049266,-0.026047,-0.026047,-0.026047,-0.052146,-0.026047,-0.045087,-0.026047,-0.050994,...,-0.045145,-0.067665,-0.026047,-0.026047,-0.036848,-0.071673,-0.026047,-0.026047,-0.026047,-0.058321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YMR Lichen,-0.026047,-0.049266,-0.026047,-0.026047,-0.026047,-0.052146,-0.026047,-0.045087,-0.026047,-0.050994,...,-0.045145,-0.067665,-0.026047,-0.026047,-0.036848,-0.071673,-0.026047,-0.026047,-0.026047,-0.058321
Yafa Zaifar,-0.026047,-0.049266,-0.026047,-0.026047,-0.026047,-0.052146,-0.026047,-0.045087,-0.026047,-0.050994,...,-0.045145,-0.067665,-0.026047,-0.026047,-0.036848,-0.071673,-0.026047,-0.026047,-0.026047,-0.058321
Yuva Sunrise,-0.026047,-0.049266,-0.026047,-0.026047,-0.026047,-0.052146,-0.026047,-0.045087,-0.026047,-0.050994,...,-0.045145,-0.067665,-0.026047,-0.026047,-0.036848,-0.071673,-0.026047,-0.026047,-0.026047,-0.058321


In [637]:


cosine_sim3 = cosine_similarity(location_df_normalized)



In [638]:
print(cosine_sim3.shape, cosine_sim2.shape, cosine_sim1.shape)

(1475, 1475) (1475, 1475) (1475, 1475)


In [639]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    cosine_sim_matrix = 30*cosine_sim1 + 8*cosine_sim3 + 2*cosine_sim2
    # cosine_sim_matrix = cosine_sim3
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim_matrix[location_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = location_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df



In [640]:
# Test the recommender function using a property name
recommend_properties_with_scores('AECS Layout RWA')

,PropertyName,SimilarityScore
0,Casagrand Aquene,26.068945
1,AM Classic,26.025994
2,Habitat Irenic,26.020929
3,Concord Heights,25.872290
4,Habilus Sunrise Apartment,25.683572
...,...,...
242,Brigade Metropolis,15.467736
243,Payal Palace,15.465014
244,Ruchira Aarna Homes,15.430311
245,Vajram Newtown 2,15.381199
